# 05 · ConvNeXt-Tiny: router, quality and organ evidential heads
One backbone; heads for modality routing (incl. `unsupported_medical` and `non_medical` via outlier exposure), image quality (synthetic degradations), and breast / cervical / ovarian suspicion (evidential).

Colab A100 ≈ 1–2 h for 20 epochs; T4 ≈ 3–4 h.

In [ ]:
# Colab setup: clone the repo and install training extras
import os
import subprocess
import sys

if "google.colab" in sys.modules and not os.path.exists("RareCare"):
    subprocess.run(["git", "clone", "https://github.com/nush1729/RareCare.git"], check=True)
    os.chdir("RareCare")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]"], check=True)
print("cwd:", os.getcwd())

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
SEED = 13

In [ ]:
!python -m training.train_vision --manifest data/manifests/vision.csv --out checkpoints/vision --epochs 20 --batch 32 --workers 4

In [ ]:
import json

import pandas as pd

h = pd.DataFrame(json.load(open("checkpoints/vision/history.json")))
h.plot(subplots=True, figsize=(10, 6));

## External validation (never seen in training): BUS-BRA (breast) and Herlev (cervical)

In [ ]:
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from sklearn.metrics import roc_auc_score

from rarecare.vision.model import load_checkpoint
from rarecare.vision.preprocess import to_model_input

model = load_checkpoint("checkpoints/vision/convnext_tiny.pt", device)

def score_folder(root, organ_key, pos_dirs, neg_dirs):
    ps, ys = [], []
    for label, dirs in ((1, pos_dirs), (0, neg_dirs)):
        for d in dirs:
            for p in sorted(Path(root, d).glob("*")):
                x = torch.from_numpy(to_model_input(Image.open(p).convert("RGB")))[None].to(device)
                with torch.no_grad():
                    a = model(x)[f"evidence_{organ_key}"][0] + 1
                ps.append(float(a[1] / a.sum())); ys.append(label)
    return roc_auc_score(ys, ps), len(ys)

# print("BUS-BRA AUROC", score_folder("data/raw/bus_bra", "breast_us", ["malignant"], ["benign"]))
# print("Herlev AUROC", score_folder("data/raw/herlev", "cervical_cytology", ["severe_dysplastic", "carcinoma_in_situ"], ["normal_superficiel", "normal_intermediate"]))

## Explanation faithfulness: Grad-CAM++ pointing game on BUSI tumour masks

In [ ]:
from rarecare.vision.gradcam import gradcam_pp

hits = n = 0
for img_p in [p for p in sorted(Path("data/raw/busi/malignant").glob("*.png")) if "_mask" not in p.stem][:200]:
    mask_p = img_p.with_name(img_p.stem + "_mask.png")
    if not mask_p.exists():
        continue
    img = Image.open(img_p).convert("RGB")
    x = torch.from_numpy(to_model_input(img))[None].to(device).requires_grad_(True)
    cam = gradcam_pp(model, x, "breast_us")
    mask = np.asarray(Image.open(mask_p).convert("L").resize((224, 224))) > 127
    y, x_ = np.unravel_index(cam.argmax(), cam.shape)
    hits += int(mask[y, x_]); n += 1
print(f"pointing game: {hits}/{n} = {hits / max(1, n):.3f}")